In [1]:
import pandas as pd
df = pd.read_csv("cholera_jhu_dataset.csv")

In [2]:
#check duplicates among Location + TL + TR
duplicate_mask = df.duplicated(subset=['Location','TL','TR'], keep='first')

# Count how many duplicates exist
duplicate_count = duplicate_mask.sum()

print("Total records:", len(df))
print("Duplicate records (same Location + TL + TR):", duplicate_count)

Total records: 232175
Duplicate records (same Location + TL + TR): 144033


In [3]:
# Mark duplicates (extra copies beyond the first) with 1
df['delete_flag'] = 0
duplicate_mask = df.duplicated(subset=['Location','TL','TR'], keep='first')
df.loc[duplicate_mask, 'delete_flag'] = 1

# Check counts
print("Duplicates marked for deletion:", df['delete_flag'].sum())
print("Remaining real records:", len(df) - df['delete_flag'].sum())

Duplicates marked for deletion: 144033
Remaining real records: 88142


In [6]:
# Filter to keep only real records (delete_flag = 0)
df_real = df[df['delete_flag'] == 0]

# Save to a new CSV file
df_real.to_csv("real_cholera_records.csv", index=False)

print("Real records saved")

Real records saved


In [7]:
df = pd.read_csv("real_cholera_records.csv")
# Count how many "::" separators each Location has
df['location_parts'] = df['Location'].str.count("::")

# Summarize counts by hierarchy depth
counts = df['location_parts'].value_counts().sort_index()

print("Records by Location format:")
for depth, count in counts.items():
    if depth == 1:
        print(f"{count} records at Country level (AFR::CMR)")
    elif depth == 2:
        print(f"{count} records at Region level (AFR::CMR::South-West)")
    elif depth == 3:
        print(f"{count} records at District level (AFR::CMR::South-West::Nguti Health District)")
    else:
        print(f"{count} records with {depth} parts")

Records by Location format:
646 records at Country level (AFR::CMR)
1156 records at Region level (AFR::CMR::South-West)
86191 records at District level (AFR::CMR::South-West::Nguti Health District)
128 records with 4 parts
21 records with 5 parts


In [8]:
# Show examples of 4-part and 5-part locations
examples_4 = df[df['location_parts'] == 4]['Location'].head(5)
examples_5 = df[df['location_parts'] == 5]['Location'].head(5)

print("\nExamples of 4-part Locations:")
for loc in examples_4:
    print(" -", loc)

print("\nExamples of 5-part Locations:")
for loc in examples_5:
    print(" -", loc)


Examples of 4-part Locations:
 - AFR::CMR::North::Mayo-Louti::Guider
 - AFR::CMR::North::Mayo-Louti::Guider
 - AFR::CMR::Far North::Diamare::Bogo
 - AFR::CMR::Far North::Logone-et-Chari::Kousseri
 - AFR::CMR::Far North::Mayo-Danay::Maga

Examples of 5-part Locations:
 - AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Guirviza
 - AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Doumo
 - AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Guirviza
 - AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Doumo
 - AFR::CMR::North::Mayo-Louti::Guider::Golombe


In [14]:
# Clean spaces around ::
df["Location_clean"] = (
    df["Location"]
    .astype(str)
    .str.replace(r"\s*::\s*", "::", regex=True)
    .str.strip()
)

# Keep only first 4 location components
def standardize_location(location):
    parts = location.split("::")
    return "::".join(parts[:4]) if len(parts) > 4 else location

df["Location_std"] = df["Location_clean"].apply(standardize_location)

In [15]:
df[df["Location_clean"].str.contains("Mayo-Louti", na=False)][
    ["Location", "Location_clean", "Location_std"]
].drop_duplicates()

,Location,Location_clean,Location_std
85724,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Guirviza,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Guirviza,AFR::CMR::North::Mayo-Louti
85725,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Doumo,AFR::CMR::North::Mayo-Louti::Mayo-Oulo::Doumo,AFR::CMR::North::Mayo-Louti
85728,AFR::CMR::North::Mayo-Louti::Guider,AFR::CMR::North::Mayo-Louti::Guider,AFR::CMR::North::Mayo-Louti
85865,AFR::CMR::North::Mayo-Louti::Figuil,AFR::CMR::North::Mayo-Louti::Figuil,AFR::CMR::North::Mayo-Louti
85868,AFR::CMR::North::Mayo-Louti::Guider::Golombe,AFR::CMR::North::Mayo-Louti::Guider::Golombe,AFR::CMR::North::Mayo-Louti
85871,AFR::CMR::North::Mayo-Louti::Mayo-Oulo,AFR::CMR::North::Mayo-Louti::Mayo-Oulo,AFR::CMR::North::Mayo-Louti


In [16]:
changed = (df["Location_clean"] != df["Location_std"]).sum()

print(f"{changed} records were standardized.")

149 records were standardized.


In [17]:
df["Location"] = df["Location_std"]

df = df.drop(columns=["Location_clean", "Location_std", "std_parts"], errors="ignore")

df.to_csv("standardized_cholera_records.csv", index=False)

In [26]:
df = pd.read_csv("standardized_cholera_records.csv")
#check duplicates among Location + TL + TR
duplicate_mask = df.duplicated(subset=['Location','TL','TR','sCh','cCh'], keep='first')

# Count how many duplicates exist
duplicate_count = duplicate_mask.sum()

print("Total records:", len(df))
print("Duplicate records (same Location + TL + TR + sCH + Cch):", duplicate_count)

Total records: 88142
Duplicate records (same Location + TL + TR + sCH + Cch): 3


In [28]:
import pandas as pd

df = pd.read_csv("standardized_cholera_records.csv")

# Remove duplicates
df = df.drop_duplicates(
    subset=['Location','TL','TR','sCh','cCh'],
    keep='first'
)

# Save cleaned file
df.to_csv(
    "standardized_cholera_records_no_duplicates.csv",
    index=False
)

print("Remaining records:", len(df))

Remaining records: 88139


In [58]:
df = pd.read_csv("standardized_cholera_records_no_duplicates.csv")
#check duplicates among Location + TL + TR +sch,cch
duplicate_mask = df.duplicated(subset=['Location','TL','TR','sCh','cCh'], keep='first')

# Count how many duplicates exist
duplicate_count = duplicate_mask.sum()

print("Total records:", len(df))
print("Duplicate records (same Location + TL + TR ):", duplicate_count)

Total records: 88139
Duplicate records (same Location + TL + TR ): 0


In [36]:
df.to_csv("Master_Dataset.csv")


In [37]:
df["Level"] = df["Location"].str.count("::")

In [40]:
df["Level"].unique()

array([3, 1, 2])

In [53]:
df = pd.read_csv("Master_Dataset.csv")

# Count how many "::" separators each Location has
df['location_parts'] = df['Location'].str.count("::")

# Summarize counts by hierarchy depth
counts = df['location_parts'].value_counts().sort_index()

print("Records by Location format:")
for depth, count in counts.items():
    if depth == 1:
        print(f"{count} records at Country level (AFR::CMR)")
    elif depth == 2:
        print(f"{count} records at Region level (AFR::CMR::South-West)")
    elif depth == 3:
        print(f"{count} records at District level (AFR::CMR::South-West::Nguti Health District)")
    else:
        print(f"{count} records with {depth} parts")

Records by Location format:
646 records at Country level (AFR::CMR)
1156 records at Region level (AFR::CMR::South-West)
86337 records at District level (AFR::CMR::South-West::Nguti Health District)


In [55]:
import pandas as pd

df = pd.read_csv("Master_Dataset.csv")

# Recalculate location_parts fresh
df["location_parts"] = df["Location"].astype(str).str.count("::")

# Check counts
print(df["location_parts"].value_counts().sort_index())

# Extract district-level records
district_df = df[df["location_parts"] == 3].copy()

print("District-level records:", len(district_df))

# Save
district_df.to_csv("district_dataset.csv", index=False)

print("Saved successfully.")

location_parts
1      646
2     1156
3    86337
Name: count, dtype: int64
District-level records: 86337
Saved successfully.


In [56]:
district_df.head()

,Unnamed: 0,Index,Location,TL,TR,deaths,sCh,cCh,CFR,reporting_date,source_index,source,confidence_weight,processing_notes,source_database,delete_flag,location_parts
0,0,1,AFR::CMR::Far North::Goulfey Health District,2017-01-09,2017-01-15,0,42,17,0.0,2017-01-15,5,UNICEF Country Programme Report (UID: 21434),0.9,"Converted from JHU database. Primary: True, Ph...",JHU,0,3
1,1,2,AFR::CMR::North-West::Batibo Health District,2014-06-09,2014-06-15,0,0,0,NaN,2014-06-15,5,UNICEF Country Programme Report (UID: 21434),0.9,"Converted from JHU database. Primary: True, Ph...",JHU,0,3
2,2,3,AFR::CMR::East::Kette Health District,2010-03-08,2010-03-14,0,0,0,NaN,2010-03-14,5,UNICEF Country Programme Report (UID: 21434),0.9,"Converted from JHU database. Primary: True, Ph...",JHU,0,3
3,3,4,AFR::CMR::Far North::Mada Health District,2015-12-14,2015-12-20,0,0,0,NaN,2015-12-20,5,UNICEF Country Programme Report (UID: 21434),0.9,"Converted from JHU database. Primary: True, Ph...",JHU,0,3
4,4,5,AFR::CMR::South::Ambam Health District,2016-01-11,2016-01-17,0,0,0,NaN,2016-01-17,5,UNICEF Country Programme Report (UID: 21434),0.9,"Converted from JHU database. Primary: True, Ph...",JHU,0,3


In [59]:
import pandas as pd

df = pd.read_csv("district_dataset.csv")

# Create Region column
df["Region"] = df["Location"].apply(
    lambda x: str(x).split("::")[2]
)

# Check results
print(df[["Location", "Region"]].head())

                                       Location      Region
0  AFR::CMR::Far North::Goulfey Health District   Far North
1  AFR::CMR::North-West::Batibo Health District  North-West
2         AFR::CMR::East::Kette Health District        East
3     AFR::CMR::Far North::Mada Health District   Far North
4        AFR::CMR::South::Ambam Health District       South


In [60]:
df.to_csv("district_dataset_with_region.csv", index=False)

In [101]:
import pandas as pd

# Load dataset
df = pd.read_csv("district_dataset_with_region.csv")

# Select required columns
selected_df = df[
    [
        "Location",
        "Region",
        "TL",
        "TR",
        "reporting_date",
        "sCh",
        "cCh",
        "deaths",
        "CFR"
    ]
]

# Save to new CSV
selected_df.to_csv(
    "FINAL district_dataset_clean.csv",
    index=False
)

print("Dataset saved successfully.")
print(selected_df.head())
print("Shape:", selected_df.shape)

Dataset saved successfully.
                                       Location      Region          TL  \
0  AFR::CMR::Far North::Goulfey Health District   Far North  2017-01-09   
1  AFR::CMR::North-West::Batibo Health District  North-West  2014-06-09   
2         AFR::CMR::East::Kette Health District        East  2010-03-08   
3     AFR::CMR::Far North::Mada Health District   Far North  2015-12-14   
4        AFR::CMR::South::Ambam Health District       South  2016-01-11   

           TR reporting_date  sCh  cCh  deaths  CFR  
0  2017-01-15     2017-01-15   42   17       0  0.0  
1  2014-06-15     2014-06-15    0    0       0  NaN  
2  2010-03-14     2010-03-14    0    0       0  NaN  
3  2015-12-20     2015-12-20    0    0       0  NaN  
4  2016-01-17     2016-01-17    0    0       0  NaN  
Shape: (86337, 9)


In [61]:
import pandas as pd

df = pd.read_csv("Master_Dataset.csv")

# Count region-level records (2 separators)
region_df = df[df["location_parts"] == 2]

print("Region-level records:", len(region_df))

Region-level records: 1156


In [63]:
region_df.to_csv("regional_data_1.csv")

In [92]:
# Load district-level data
df = pd.read_csv("district_dataset_with_region.csv")

# Group districts into regions by reporting date
regional_from_district = (
    df.groupby(["Region","TL","TR", "reporting_date"], as_index=False)
      .agg({
          "sCh": "sum",
          "cCh": "sum",
          "deaths": "sum"
      })
)

In [95]:
# Recalculate CFR
regional_from_district["CFR"] = (
    regional_from_district["deaths"] / regional_from_district["cCh"]
) * 100

regional_from_district.loc[
    regional_from_district["cCh"] == 0, "CFR"
] = 0

# Save
regional_from_district.to_csv("regional_data_2.csv",index=False)
print("Rows:", len(regional_from_district))
print("Regions:", regional_from_district["Region"].nunique())

Rows: 4949
Regions: 10


In [90]:
df = pd.read_csv("regional_data_1.csv")

df["Region"] = (
    df["Location"]
    .astype(str)
    .str.split("::")
    .str[-1]
)

In [91]:
df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Index,Location,TL,TR,deaths,sCh,cCh,CFR,reporting_date,source_index,source,confidence_weight,processing_notes,source_database,delete_flag,location_parts,Region
0,0,85719,85719,85721,AFR::CMR::Far North,2018-10-31,2018-11-02,0,91,42,0.0,2018-11-02,18,WHO AFRO Surveillance Report (UID: 21027),0.97,"Converted from JHU database. Primary: True, Ph...",JHU,0,2,Far North
1,1,85720,85720,85722,AFR::CMR::North,2018-10-31,2018-11-02,0,38,14,0.0,2018-11-02,18,WHO AFRO Surveillance Report (UID: 21027),0.97,"Converted from JHU database. Primary: True, Ph...",JHU,0,2,North
2,2,85721,85721,85723,AFR::CMR::Centre,2018-08-27,2018-11-02,0,0,0,NaN,2018-11-02,18,WHO AFRO Surveillance Report (UID: 21027),0.97,"Converted from JHU database. Primary: True, Ph...",JHU,0,2,Centre
3,3,85722,85722,85724,AFR::CMR::Littoral,2018-10-11,2018-11-02,0,0,0,NaN,2018-11-02,18,WHO AFRO Surveillance Report (UID: 21027),0.97,"Converted from JHU database. Primary: True, Ph...",JHU,0,2,Littoral
4,4,85731,85731,85735,AFR::CMR::Adamaoua,2010-05-06,2010-12-31,0,1,1,0.0,2010-12-31,34,JHU Cholera Database (UID: 182),0.95,"Converted from JHU database. Primary: True, Ph...",JHU,0,2,Adamaoua


In [77]:
print(
    df[
        ["Location", "Region"]
    ].head(10)
)

              Location     Region
0  AFR::CMR::Far North  Far North
1      AFR::CMR::North      North
2     AFR::CMR::Centre     Centre
3   AFR::CMR::Littoral   Littoral
4   AFR::CMR::Adamaoua   Adamaoua
5     AFR::CMR::Centre     Centre
6       AFR::CMR::East       East
7  AFR::CMR::Far North  Far North
8   AFR::CMR::Littoral   Littoral
9      AFR::CMR::North      North


In [79]:
df.to_csv("regional_data_1.csv")

In [96]:
regional_data_1 = pd.read_csv("regional_data_1.csv")
regional_data_2 = pd.read_csv("regional_data_2.csv")
print("Regional Data 1:", len(regional_data_1))
print("Regional Data 2:", len(regional_data_2))

Regional Data 1: 1156
Regional Data 2: 4949


In [97]:
import pandas as pd

# Stack both datasets
combined = pd.concat(
    [regional_data_1, regional_data_2],
    ignore_index=True
)

# Remove duplicates
combined = combined.drop_duplicates(
    subset=["Region", "reporting_date"],
    keep="first"
)

print("Final rows:", len(combined))

# Save
combined.to_csv(
    "regional_master_dataset.csv",
    index=False
)

Final rows: 5295


In [98]:
import pandas as pd

df = pd.read_csv("regional_master_dataset.csv")

# Keep only needed columns
df = df[
    [
        "Region",
        "reporting_date",
        "TL",
        "TR",
        "sCh",
        "cCh",
        "deaths",
        "CFR"
    ]
]

# Save
df.to_csv(
    "regional_master_dataset_clean.csv",
    index=False
)

print(df.head())
print(df.shape)

      Region reporting_date          TL          TR  sCh  cCh  deaths  CFR
0  Far North     2018-11-02  2018-10-31  2018-11-02   91   42       0  0.0
1      North     2018-11-02  2018-10-31  2018-11-02   38   14       0  0.0
2     Centre     2018-11-02  2018-08-27  2018-11-02    0    0       0  NaN
3   Littoral     2018-11-02  2018-10-11  2018-11-02    0    0       0  NaN
4   Adamaoua     2010-12-31  2010-05-06  2010-12-31    1    1       0  0.0
(5295, 8)


In [99]:
import pandas as pd
import requests
import time

df = pd.read_csv("regional_master_dataset_clean.csv")

# Convert dates
df["TL"] = pd.to_datetime(df["TL"])
df["TR"] = pd.to_datetime(df["TR"])

# Approximate regional coordinates for Cameroon
region_coords = {
    "Adamaoua": (7.32, 13.58),
    "Centre": (4.75, 11.83),
    "East": (4.98, 14.30),
    "Far North": (10.59, 14.32),
    "Littoral": (4.05, 9.70),
    "North": (8.65, 13.90),
    "North-West": (6.33, 10.40),
    "South": (2.93, 11.15),
    "South-West": (4.60, 9.30),
    "West": (5.50, 10.50)
}

def fetch_nasa_average(region, start_date, end_date):
    try:
        lat, lon = region_coords[region]

        start = pd.to_datetime(start_date).strftime("%Y%m%d")
        end = pd.to_datetime(end_date).strftime("%Y%m%d")

        url = "https://power.larc.nasa.gov/api/temporal/daily/point"

        params = {
            "parameters": "PRECTOTCORR,T2M,RH2M",
            "community": "AG",
            "longitude": lon,
            "latitude": lat,
            "start": start,
            "end": end,
            "format": "JSON"
        }

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()["properties"]["parameter"]

        rainfall = pd.Series(data["PRECTOTCORR"]).astype(float).mean()
        temperature = pd.Series(data["T2M"]).astype(float).mean()
        humidity = pd.Series(data["RH2M"]).astype(float).mean()

        return rainfall, temperature, humidity

    except Exception as e:
        print(f"Failed for {region}, {start_date} to {end_date}: {e}")
        return None, None, None

In [100]:
# Avoid repeated API calls by fetching each Region + TL + TR once
unique_periods = df[["Region", "TL", "TR"]].drop_duplicates()

results = []

for _, row in unique_periods.iterrows():
    region = row["Region"]

    if region not in region_coords:
        continue

    rainfall, temperature, humidity = fetch_nasa_average(
        region,
        row["TL"],
        row["TR"]
    )

    results.append({
        "Region": region,
        "TL": row["TL"],
        "TR": row["TR"],
        "rainfall_avg": rainfall,
        "temperature_avg": temperature,
        "humidity_avg": humidity
    })

    time.sleep(0.5)  # avoid overwhelming API

env_df = pd.DataFrame(results)

# Merge environmental data into cholera dataset
final_df = df.merge(
    env_df,
    on=["Region", "TL", "TR"],
    how="left"
)

final_df.to_csv("regional_cholera_environment_dataset.csv", index=False)

print("Done.")
print(final_df.head())

Failed for South-West, 2018-12-10 00:00:00 to 2018-12-16 00:00:00: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/daily/point?parameters=PRECTOTCORR%2CT2M%2CRH2M&community=AG&longitude=9.3&latitude=4.6&start=20181210&end=20181216&format=JSON (Caused by NameResolutionError("HTTPSConnection(host='power.larc.nasa.gov', port=443): Failed to resolve 'power.larc.nasa.gov' ([Errno 11001] getaddrinfo failed)"))
Failed for North-West, 2012-10-01 00:00:00 to 2012-10-07 00:00:00: HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/daily/point?parameters=PRECTOTCORR%2CT2M%2CRH2M&community=AG&longitude=10.4&latitude=6.33&start=20121001&end=20121007&format=JSON (Caused by NameResolutionError("HTTPSConnection(host='power.larc.nasa.gov', port=443): Failed to resolve 'power.larc.nasa.gov' ([Errno 11001] getaddrinfo failed)"))
Done.
      Region reporting_date         TL         TR  sCh  cCh  dea

In [103]:
df=pd.read_csv("FINAL regional_cholera_environment_dataset.csv")
df.describe()

,sCh,cCh,deaths,CFR,rainfall_avg,temperature_avg,humidity_avg
count,5280.000000,5280.000000,5280.000000,4485.000000,5278.000000,5278.000000,5278.000000
mean,129.252083,41.885606,0.885038,0.997482,4.871166,24.165966,75.249818
std,275.183394,107.231920,3.897221,1.568770,5.483518,2.845871,19.991426
min,0.000000,0.000000,0.000000,0.000000,0.000000,16.328571,7.998571
25%,0.000000,0.000000,0.000000,0.000000,0.387500,22.240000,69.474643
50%,58.000000,14.000000,0.000000,0.000000,3.678571,23.842143,83.830714
75%,159.000000,51.000000,1.000000,1.886792,7.185714,25.785357,88.350000
max,7633.000000,3920.000000,146.000000,20.000000,79.830000,35.200000,95.070000


In [105]:
cols = [
    "cCh",
    "sCh",
    "deaths",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg"
]

print(df[cols].corr())

                      cCh       sCh    deaths  rainfall_avg  temperature_avg  \
cCh              1.000000  0.893943  0.857560      0.163103         0.059041   
sCh              0.893943  1.000000  0.810257      0.165795         0.078978   
deaths           0.857560  0.810257  1.000000      0.091359         0.053797   
rainfall_avg     0.163103  0.165795  0.091359      1.000000        -0.259916   
temperature_avg  0.059041  0.078978  0.053797     -0.259916         1.000000   
humidity_avg     0.101856  0.107981  0.055262      0.508414        -0.483778   

                 humidity_avg  
cCh                  0.101856  
sCh                  0.107981  
deaths               0.055262  
rainfall_avg         0.508414  
temperature_avg     -0.483778  
humidity_avg         1.000000  


In [108]:
import statsmodels.api as sm
# Select variables
X = df[[
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg"
]]

y = df["cCh"]

# Remove missing values
data = pd.concat([X, y], axis=1).dropna()

X = data[[
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg"
]]

y = data["cCh"]

# Add intercept
X = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X).fit()

# Display full regression results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    cCh   R-squared:                       0.043
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     79.00
Date:                Fri, 29 May 2026   Prob (F-statistic):           5.47e-50
Time:                        23:32:22   Log-Likelihood:                -32047.
No. Observations:                5278   AIC:                         6.410e+04
Df Residuals:                    5274   BIC:                         6.413e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            -141.9046     17.982     

In [111]:
import pandas as pd

df = pd.read_csv("Master_Dataset.csv")

# Count region-level records (1 separators)
country_df = df[df["location_parts"] == 1]

print("country-level records:", len(region_df))

country-level records: 1156


In [112]:
country_df.to_csv("country_level_data_1.csv")

In [115]:
df = pd.read_csv("FINAL regional_cholera_environment_dataset.csv")


In [120]:
# Convert reporting_date to datetime
df['reporting_date'] = pd.to_datetime(df['reporting_date'])

# Aggregate sums for cases and deaths, averages for climate variables
agg_dict = {
    'sCh': 'sum',
    'cCh': 'sum',
    'deaths': 'sum'
}

# Group by reporting_date (and TL/TR if you want to keep them)
country_df = df.groupby(['reporting_date', 'TL', 'TR'], as_index=False).agg(agg_dict)

# Compute CFR at country level
country_df['CFR'] = (country_df['deaths'] / country_df['cCh'].replace(0, pd.NA)) * 100

# Add Region column
country_df['Region'] = 'Cameroon'

# Save to CSV
country_df.to_csv("country_level_data_2.csv", index=False)

print(country_df.head())


  reporting_date         TL         TR   sCh  cCh  deaths       CFR    Region
0     2010-01-10   1/4/2010  1/10/2010  1049  256       3  1.171875  Cameroon
1     2010-01-17  1/11/2010  1/17/2010   538  148       0       0.0  Cameroon
2     2010-01-24  1/18/2010  1/24/2010   322  103       0       0.0  Cameroon
3     2010-01-31  1/25/2010  1/31/2010   346   94       1   1.06383  Cameroon
4     2010-02-07   2/1/2010   2/7/2010   633  160       3     1.875  Cameroon


In [125]:
len(df1)


646

In [126]:
len(df2)

722

In [123]:
import pandas as pd

# Load both files
df1 = pd.read_csv("country_level_data_1.csv")
df2 = pd.read_csv("country_level_data_2.csv")
# Keep only the columns you want
cols = ["TL", "TR", "reporting_date", "sCh", "cCh", "CFR"]
df1 = df1[cols]
df2 = df2[cols]

# Concatenate row-wise
combined = pd.concat([df1, df2], ignore_index=True)

# Save to one final file
combined.to_csv("country_level_data_final.csv", index=False)

print("Final file saved as country_level_data_final.csv")
print(combined.head())

Final file saved as country_level_data_final.csv
           TL          TR reporting_date    sCh  cCh   CFR
0  2020-01-01  2020-09-30     2020-09-30      0    0   NaN
1  2018-05-18  2018-11-02     2018-11-02    745   53  7.55
2  2018-05-18  2018-07-05     2018-07-05     26    8  0.00
3  2010-05-06  2010-12-31     2010-12-31  10759    0   NaN
4  2011-01-01  2011-09-22     2011-09-22  17121    0   NaN


In [128]:
import pandas as pd

# Load dataset
df = pd.read_csv("density/WB_WDI_EN_POP_DNST.csv")

# Filter Cameroon records
df_cmr = df[df['REF_AREA'] == 'CMR']

# Keep only important columns
df_cmr = df_cmr[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']]

# Rename columns for clarity
df_cmr = df_cmr.rename(columns={
    'TIME_PERIOD': 'Year',
    'OBS_VALUE': 'Population_Density'
})

# Check result
print(df_cmr.head())
print(f"Number of records for Cameroon: {len(df_cmr)}")

# Save cleaned file
df_cmr.to_csv("cmr_DENSITY.csv", index=False)

    REF_AREA  Year  Population_Density
512      CMR  1998           29.922913
716      CMR  2011           42.800668
752      CMR  2012           44.021766
774      CMR  2017           51.043136
831      CMR  1999           30.727380
Number of records for Cameroon: 63


In [130]:
import pandas as pd

df = pd.read_csv("water&sanitation/washdash-download.csv")

df_clean = df[
    (df["ISO3"] == "CMR") &
    (df["Residence Type"] == "total")
][[
    "Year",
    "Service Type",
    "Service level",
    "Coverage"
]]

df_clean.to_csv("cmr_water_sanitation.csv", index=False)

print(df_clean.head())

    Year    Service Type    Service level  Coverage
0   2011  Drinking water   At least basic  57.99071
3   2011      Sanitation   At least basic  34.00940
6   2011  Drinking water  Limited service   9.81510
9   2011      Sanitation  Limited service  17.49131
12  2011      Sanitation  Open defecation   2.41003


In [1]:
import pandas as pd

df = pd.read_csv("FINAL regional_cholera_environment_dataset.csv")
print(df.shape)

(5280, 11)


In [2]:
# 2. Convert date
df["reporting_date"] = pd.to_datetime(df["reporting_date"], errors="coerce")

# 3. Convert case columns to numeric
df["sCh"] = pd.to_numeric(df["sCh"], errors="coerce").fillna(0)
df["cCh"] = pd.to_numeric(df["cCh"], errors="coerce").fillna(0)

# 4. Create total cases
df["cases"] = df["sCh"] + df["cCh"]

# 5. Create month column
df["month_date"] = df["reporting_date"].dt.to_period("M").dt.to_timestamp()

# 6. Aggregate to monthly regional level
monthly = (
    df.groupby(["Region", "month_date"])
    .agg({
        "cases": "sum",
        "rainfall_avg": "mean",
        "temperature_avg": "mean",
        "humidity_avg": "mean"
    })
    .reset_index()
)

# 7. Sort data
monthly = monthly.sort_values(["Region", "month_date"])

# 8. Check result
print(monthly.head(20))
print(monthly.shape)
print(monthly["cases"].describe())
print(monthly["Region"].nunique())

      Region month_date  cases  rainfall_avg  temperature_avg  humidity_avg
0   Adamaoua 2010-01-01     19      0.028214        22.468571     40.163929
1   Adamaoua 2010-02-01     47      0.039643        24.665000     43.458214
2   Adamaoua 2010-03-01    154      1.157500        25.286429     47.182500
3   Adamaoua 2010-04-01    378      3.931429        25.590357     63.309286
4   Adamaoua 2010-05-01     86      4.798286        24.232857     77.565714
5   Adamaoua 2010-06-01    126      6.778571        23.131071     79.708214
6   Adamaoua 2010-07-01    479      8.108214        21.512857     85.185000
7   Adamaoua 2010-08-01    204     11.916857        21.222857     86.643714
8   Adamaoua 2010-09-01    120      5.096786        21.299643     86.566071
9   Adamaoua 2010-10-01     62      5.518571        21.852571     84.123429
10  Adamaoua 2010-11-01    128      1.096786        22.248214     73.478929
11  Adamaoua 2010-12-01     82      1.099260        21.084288     58.553569
12  Adamaoua

In [3]:
monthly.groupby("Region").size().sort_values()

Region
Adamaoua      113
East          113
North-West    113
West          121
North         127
Far North     127
South-West    129
Littoral      129
South         130
Centre        135
dtype: int64

In [158]:
print("Total rows:", len(monthly))

print("Zero case rows:",
      (monthly["cases"] == 0).sum())

print("Percent zeros:",
      round(
          (monthly["cases"] == 0).mean() * 100,
          2
      ),
      "%"
)

Total rows: 1237
Zero case rows: 115
Percent zeros: 9.3 %


In [159]:
monthly = monthly.sort_values(
    ["Region", "month_date"]
)

monthly["cases_lag1"] = (
    monthly.groupby("Region")["cases"]
    .shift(1)
)

monthly["cases_lag2"] = (
    monthly.groupby("Region")["cases"]
    .shift(2)
)

monthly["cases_lag3"] = (
    monthly.groupby("Region")["cases"]
    .shift(3)
)

In [160]:
monthly["future_cases"] = (
    monthly.groupby("Region")["cases"]
    .shift(-1)
)

In [161]:
monthly = monthly.dropna()

In [162]:
print(
    monthly[
        [
            "Region",
            "month_date",
            "cases",
            "cases_lag1",
            "cases_lag2",
            "cases_lag3",
            "future_cases"
        ]
    ].head(20)
)

      Region month_date  cases  cases_lag1  cases_lag2  cases_lag3  \
3   Adamaoua 2010-04-01    378       154.0        47.0        19.0   
4   Adamaoua 2010-05-01     86       378.0       154.0        47.0   
5   Adamaoua 2010-06-01    126        86.0       378.0       154.0   
6   Adamaoua 2010-07-01    479       126.0        86.0       378.0   
7   Adamaoua 2010-08-01    204       479.0       126.0        86.0   
8   Adamaoua 2010-09-01    120       204.0       479.0       126.0   
9   Adamaoua 2010-10-01     62       120.0       204.0       479.0   
10  Adamaoua 2010-11-01    128        62.0       120.0       204.0   
11  Adamaoua 2010-12-01     82       128.0        62.0       120.0   
12  Adamaoua 2011-01-01    161        82.0       128.0        62.0   
13  Adamaoua 2011-02-01      0       161.0        82.0       128.0   
14  Adamaoua 2011-03-01     75         0.0       161.0        82.0   
15  Adamaoua 2011-04-01    209        75.0         0.0       161.0   
16  Adamaoua 2011-05

In [163]:
print(
    monthly[
        [
            "cases",
            "cases_lag1",
            "cases_lag2",
            "cases_lag3",
            "future_cases"
        ]
    ].corr()
)

                 cases  cases_lag1  cases_lag2  cases_lag3  future_cases
cases         1.000000    0.636567    0.499112    0.333249      0.630380
cases_lag1    0.636567    1.000000    0.683901    0.536751      0.486954
cases_lag2    0.499112    0.683901    1.000000    0.683372      0.324626
cases_lag3    0.333249    0.536751    0.683372    1.000000      0.200832
future_cases  0.630380    0.486954    0.324626    0.200832      1.000000


In [164]:
import numpy as np

monthly["future_cases_log"] = np.log1p(
    monthly["future_cases"]
)

print(
    monthly["future_cases_log"].describe()
)

count    1197.000000
mean        5.482958
std         2.133782
min         0.000000
25%         4.867534
50%         6.033086
75%         6.824374
max         9.627602
Name: future_cases_log, dtype: float64


In [165]:
print(monthly["future_cases"].describe())
print(monthly["future_cases_log"].describe())

count     1197.000000
mean       742.304929
std       1143.431341
min          0.000000
25%        129.000000
50%        416.000000
75%        919.000000
max      15177.000000
Name: future_cases, dtype: float64
count    1197.000000
mean        5.482958
std         2.133782
min         0.000000
25%         4.867534
50%         6.033086
75%         6.824374
max         9.627602
Name: future_cases_log, dtype: float64


In [166]:
features = [
    "cases",
    "cases_lag1",
    "cases_lag2",
    "cases_lag3",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg",
    "month",
    "Region_enc"
]

In [167]:
import numpy as np

monthly["future_cases_log"] = np.log1p(
    monthly["future_cases"]
)

y = monthly["future_cases_log"]

In [173]:
print(monthly.shape)
print(df.shape)

(1197, 11)
(5280, 13)


In [174]:
train = monthly[
    monthly["month_date"] < "2021-01-01"
]

test = monthly[
    monthly["month_date"] >= "2021-01-01"
]

print(train.shape)
print(test.shape)

print(train["month_date"].min())
print(train["month_date"].max())

print(test["month_date"].min())
print(test["month_date"].max())

(1146, 11)
(51, 11)
2010-04-01 00:00:00
2020-12-01 00:00:00
2021-10-01 00:00:00
2022-12-01 00:00:00


In [175]:
monthly[
    (monthly["month_date"] >= "2021-01-01") &
    (monthly["month_date"] < "2021-10-01")
].shape

(0, 11)

In [176]:
monthly[
    (monthly["month_date"] >= "2021-01-01") &
    (monthly["month_date"] < "2021-10-01")
][["Region","month_date"]].head(20)

,Region,month_date


In [177]:
features = [
    "cases",
    "cases_lag1",
    "cases_lag2",
    "cases_lag3",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg"
]

print(
    monthly[
        features + ["future_cases"]
    ].corr()["future_cases"]
    .sort_values(ascending=False)
)

future_cases       1.000000
cases              0.630380
cases_lag1         0.486954
cases_lag2         0.324626
rainfall_avg       0.240813
cases_lag3         0.200832
temperature_avg    0.150885
humidity_avg       0.120197
Name: future_cases, dtype: float64


In [178]:
q1 = monthly["future_cases"].quantile(0.33)
q2 = monthly["future_cases"].quantile(0.66)

print(q1, q2)

208.04000000000002 696.08


In [179]:
def risk_label(x):

    if x <= q1:
        return 0   # Low

    elif x <= q2:
        return 1   # Medium

    return 2       # High

monthly["future_risk"] = (
    monthly["future_cases"]
    .apply(risk_label)
)

In [180]:
from xgboost import XGBClassifier

In [185]:
region_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

print(region_mapping)

{'Adamaoua': np.int64(0), 'Centre': np.int64(1), 'East': np.int64(2), 'Far North': np.int64(3), 'Littoral': np.int64(4), 'North': np.int64(5), 'North-West': np.int64(6), 'South': np.int64(7), 'South-West': np.int64(8), 'West': np.int64(9)}


In [186]:
import joblib

joblib.dump(le, "region_encoder.pkl")

['region_encoder.pkl']

In [182]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Encode region if not already done
le = LabelEncoder()
monthly["Region_enc"] = le.fit_transform(monthly["Region"])

# Create month feature if not already done
monthly["month"] = monthly["month_date"].dt.month

# Features
features = [
    "cases",
    "cases_lag1",
    "cases_lag2",
    "cases_lag3",
    "rainfall_avg",
    "temperature_avg",
    "humidity_avg",
    "month",
    "Region_enc"
]

# Train-test split
train = monthly[monthly["month_date"] < "2021-01-01"]
test = monthly[monthly["month_date"] >= "2021-01-01"]

X_train = train[features]
X_test = test[features]

y_train = train["future_risk"]
y_test = test["future_risk"]

# Train XGBoost classifier
xgb_clf = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_clf.fit(X_train, y_train)

# Predict
pred = xgb_clf.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

Accuracy: 0.6274509803921569
              precision    recall  f1-score   support

           0       0.83      0.75      0.79        32
           1       0.27      0.30      0.29        10
           2       0.45      0.56      0.50         9

    accuracy                           0.63        51
   macro avg       0.52      0.54      0.52        51
weighted avg       0.65      0.63      0.64        51

[[24  6  2]
 [ 3  3  4]
 [ 2  2  5]]


In [183]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    )
}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    print("\n", name)
    print("Accuracy:", accuracy_score(y_test, pred))
    print(classification_report(y_test, pred))
    print(confusion_matrix(y_test, pred))


 Random Forest
Accuracy: 0.6274509803921569
              precision    recall  f1-score   support

           0       0.81      0.81      0.81        32
           1       0.20      0.20      0.20        10
           2       0.44      0.44      0.44         9

    accuracy                           0.63        51
   macro avg       0.49      0.49      0.49        51
weighted avg       0.63      0.63      0.63        51

[[26  5  1]
 [ 4  2  4]
 [ 2  3  4]]
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001105 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 1146, number of used features: 9
[LightGBM] [Info] Start training from score -1.149630
[LightGBM] [Info] Start training from score -1.090790
[LightGBM] [Info] Start training from score -1.057581
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

In [184]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": features,
    "Importance": xgb_clf.feature_importances_
})

print(
    importance.sort_values(
        "Importance",
        ascending=False
    )
)

           Feature  Importance
0            cases    0.327135
7            month    0.165241
2       cases_lag2    0.089646
1       cases_lag1    0.081920
5  temperature_avg    0.070940
3       cases_lag3    0.070677
6     humidity_avg    0.066428
8       Region_enc    0.064527
4     rainfall_avg    0.063487
